# Task Subagent Token Overhead

**Purpose:** Task subagents are the easy default for parallel work in Claude Code —
no subprocess management, no prompt engineering, just `Task(prompt)`. But direct `claude -p`
is far more capable: full control over prompt, model, session, permissions, and subprocess-level
parallelism. This notebook measures whether that power also comes with token savings.

| # | Question | Proof method |
|---|----------|--------------|
| 1 | Does direct `claude -p` use fewer tokens than Task delegation? | Compare token counts across 5 runs per pattern |
| 2 | Are the savings fixed or do they scale with complexity? | Three complexity tiers (no tools, 1 tool, multi-step) |
| 3 | Where do Task subagent tokens go? | Strategy B: parent on haiku, subagent on sonnet → separate modelUsage entries |
| 4 | What is the cost difference per invocation? | Haiku 4.5 pricing applied to token delta |

## Background

### Two invocation patterns

- **Pattern A (Direct):** `claude -p "Do X"` — the agent receives the prompt and acts on it directly
- **Pattern B (Task):** `claude -p "Use the Task tool to: Do X"` — the parent agent delegates
  via Task tool, which spawns a subagent that does the actual work

### Task delegation overhead steps

When the parent delegates via Task:

1. Parent reads its system prompt (input tokens)
2. Parent reads the user prompt instructing Task delegation (input tokens)
3. Parent generates the Task tool call JSON (output tokens)
4. **Subagent** reads its own system prompt + task description (input tokens)
5. **Subagent** does the work and returns a result (output tokens)
6. Parent reads the subagent result, formulates final response (input + output tokens)

Steps 1-3 and 6 are pure overhead that direct invocation avoids.

### The model-merging problem

When parent and subagent both run the same model (e.g., haiku), `modelUsage` in the
output JSON merges their token counts into one entry. We can measure *total* overhead
as a delta between patterns, but cannot decompose parent vs subagent.

**Strategy B** solves this by running the subagent on a different model (sonnet),
producing separate `modelUsage` entries.

### Pricing reference (Haiku 4.5)

| Metric | Price per 1M tokens |
|--------|--------------------|
| Input tokens | $1.00 |
| Output tokens | $5.00 |
| Cache read input tokens | $0.10 |

In [ ]:
import json
import subprocess
import uuid
import statistics
from pathlib import Path
from lib import run_claude

N_RUNS = 5


def extract_usage(cr):
    """Extract token usage from a ClaudeResult."""
    output = cr.output if isinstance(cr.output, dict) else {}
    model_usage = output.get("modelUsage", {})
    num_turns = output.get("num_turns", 0)

    total_input = sum(u.get("inputTokens", 0) for u in model_usage.values())
    total_output = sum(u.get("outputTokens", 0) for u in model_usage.values())
    total_cache = sum(u.get("cacheReadInputTokens", 0) for u in model_usage.values())

    return {
        "inputTokens": total_input,
        "outputTokens": total_output,
        "cacheReadInputTokens": total_cache,
        "total": total_input + total_output + total_cache,
        "num_turns": num_turns,
        "models": list(model_usage.keys()),
        "model_usage": model_usage,
    }


def run_experiment(prompt, n=N_RUNS, label=""):
    """Run claude -p N times and collect usage stats."""
    results = []
    for i in range(n):
        cr = run_claude(prompt)
        usage = extract_usage(cr)
        usage["run"] = i + 1
        usage["returncode"] = cr.returncode
        results.append(usage)
        status = "OK" if cr.returncode == 0 else "FAIL"
        print(f"  [{label}] Run {i+1}/{n}: {status} — "
              f"in={usage['inputTokens']} out={usage['outputTokens']} "
              f"cache={usage['cacheReadInputTokens']} turns={usage['num_turns']}")
    return results


def summarize(results, metric):
    """Compute mean/stdev/min/max for a metric across runs."""
    values = [r[metric] for r in results]
    return {
        "mean": statistics.mean(values),
        "stdev": statistics.stdev(values) if len(values) > 1 else 0,
        "min": min(values),
        "max": max(values),
        "n": len(values),
    }


def compare(direct_results, task_results, tier):
    """Print comparison table and return summary dict."""
    metrics = ["inputTokens", "outputTokens", "cacheReadInputTokens", "total", "num_turns"]
    labels = ["Input", "Output", "Cache Read", "Total", "Turns"]

    print(f"\n{'='*74}")
    print(f"  {tier.upper()} TIER — Direct vs Task Delegation")
    print(f"{'='*74}")
    print(f"  {'Metric':<12} {'Direct':>18} {'Task':>18} {'Delta':>10} {'Overhead':>10}")
    print(f"  {'-'*12} {'-'*18} {'-'*18} {'-'*10} {'-'*10}")

    summary = {}
    for metric, label in zip(metrics, labels):
        d = summarize(direct_results, metric)
        t = summarize(task_results, metric)
        delta = t["mean"] - d["mean"]
        pct = (delta / d["mean"] * 100) if d["mean"] > 0 else float('inf')
        print(f"  {label:<12} {d['mean']:>8.0f} ± {d['stdev']:>5.0f}"
              f" {t['mean']:>8.0f} ± {t['stdev']:>5.0f}"
              f" {delta:>+10.0f} {pct:>+9.1f}%")
        summary[metric] = {"direct": d, "task": t, "delta": delta, "pct": pct}

    return summary


def cost_estimate(summary):
    """Estimate cost using Haiku 4.5 pricing from a compare() summary."""
    # $1/M input, $5/M output, $0.10/M cache read
    result = {}
    for pattern in ["direct", "task"]:
        inp = summary["inputTokens"][pattern]["mean"] / 1_000_000 * 1.0
        out = summary["outputTokens"][pattern]["mean"] / 1_000_000 * 5.0
        cache = summary["cacheReadInputTokens"][pattern]["mean"] / 1_000_000 * 0.10
        result[pattern] = {"input": inp, "output": out, "cache_read": cache, "total": inp + out + cache}
    result["overhead"] = result["task"]["total"] - result["direct"]["total"]
    return result


# Storage for cross-tier analysis
ALL_RESULTS = {}   # {tier: {"direct": [...], "task": [...]}}
ALL_SUMMARIES = {} # {tier: compare() output}

print("Setup ready.")

## Experiment design

| Tier | Direct prompt | Task prompt | Expected behavior |
|------|--------------|-------------|-------------------|
| Simple | "What is 2+2?" | "Use the Task tool to: What is 2+2?" | No tools — isolates pure delegation overhead |
| Medium | "Read research/lib.py and count function defs" | "Use the Task tool to: Read research/lib.py..." | 1 tool call (Read) — adds tool-use overhead |
| Complex | "Find .py files via Glob, count defs via Grep" | "Use the Task tool to: Find .py files..." | Multi-step (Glob + Grep) — tests overhead scaling |

Each experiment: 5 runs per pattern, haiku model, default permission mode.

### Strategy A (merged models)

Both parent and subagent run on haiku. `modelUsage` merges their tokens.
Overhead = `task_total - direct_total`.

### Strategy B (separated models)

Parent on haiku, subagent on sonnet. `modelUsage` shows two entries.
Allows decomposing parent overhead vs subagent work.

In [ ]:
PROMPTS = {
    "simple": {
        "direct": "What is 2+2? Reply with just the number.",
        "task": (
            "Use the Task tool to answer this question. "
            "Set subagent_type to 'general-purpose' and the prompt to: "
            "'What is 2+2? Reply with just the number.' "
            "Then return whatever the task returns."
        ),
    },
    "medium": {
        "direct": (
            "Read the file research/lib.py and count the number of function definitions "
            "(lines starting with 'def '). Reply with just the count."
        ),
        "task": (
            "Use the Task tool to perform this task. "
            "Set subagent_type to 'general-purpose' and the prompt to: "
            "'Read the file research/lib.py and count the number of function definitions "
            "(lines starting with def). Reply with just the count.' "
            "Then return whatever the task returns."
        ),
    },
    "complex": {
        "direct": (
            "Find all .py files under the research/ directory using Glob, "
            "then use Grep to count how many lines match 'def ' across all of them. "
            "Reply with the file list and the total count of function definitions."
        ),
        "task": (
            "Use the Task tool to perform this task. "
            "Set subagent_type to 'general-purpose' and the prompt to: "
            "'Find all .py files under the research/ directory using Glob, "
            "then use Grep to count how many lines match def across all of them. "
            "Reply with the file list and the total count of function definitions.' "
            "Then return whatever the task returns."
        ),
    },
}

for tier, prompts in PROMPTS.items():
    print(f"\n{tier}:")
    for pattern, prompt in prompts.items():
        print(f"  {pattern}: {prompt[:80]}...")

---

## Strategy A: Merged Models (both haiku)

Parent and subagent both run on haiku. `modelUsage` merges their token counts.
We measure total overhead as the delta between direct and task-delegated patterns.

In [ ]:
# TEST 1: Simple tier — "What is 2+2?" (no tools)
# Isolates pure delegation overhead: system prompt + tool framing + result processing

print("TEST 1: SIMPLE TIER")
print("="*40)

print("\nPattern A (Direct):")
simple_direct = run_experiment(PROMPTS["simple"]["direct"], label="simple/direct")

print("\nPattern B (Task):")
simple_task = run_experiment(PROMPTS["simple"]["task"], label="simple/task")

ALL_RESULTS["simple"] = {"direct": simple_direct, "task": simple_task}
ALL_SUMMARIES["simple"] = compare(simple_direct, simple_task, "simple")

In [ ]:
# TEST 2: Medium tier — Read a file and count function definitions (1 tool call)
# Adds tool-use overhead on top of delegation overhead

print("TEST 2: MEDIUM TIER")
print("="*40)

print("\nPattern A (Direct):")
medium_direct = run_experiment(PROMPTS["medium"]["direct"], label="medium/direct")

print("\nPattern B (Task):")
medium_task = run_experiment(PROMPTS["medium"]["task"], label="medium/task")

ALL_RESULTS["medium"] = {"direct": medium_direct, "task": medium_task}
ALL_SUMMARIES["medium"] = compare(medium_direct, medium_task, "medium")

In [ ]:
# TEST 3: Complex tier — Glob + Grep multi-step (multiple tool calls)
# Tests whether overhead scales with task complexity

print("TEST 3: COMPLEX TIER")
print("="*40)

print("\nPattern A (Direct):")
complex_direct = run_experiment(PROMPTS["complex"]["direct"], label="complex/direct")

print("\nPattern B (Task):")
complex_task = run_experiment(PROMPTS["complex"]["task"], label="complex/task")

ALL_RESULTS["complex"] = {"direct": complex_direct, "task": complex_task}
ALL_SUMMARIES["complex"] = compare(complex_direct, complex_task, "complex")

In [ ]:
# Strategy A cross-tier summary
# Check: is overhead fixed (constant across tiers) or scaling (grows with complexity)?

print("STRATEGY A — CROSS-TIER OVERHEAD SUMMARY")
print("=" * 74)
print(f"  {'Tier':<10} {'Direct Total':>14} {'Task Total':>14} {'Overhead':>14} {'Overhead %':>12}")
print(f"  {'-'*10} {'-'*14} {'-'*14} {'-'*14} {'-'*12}")

overheads = []
for tier in ["simple", "medium", "complex"]:
    s = ALL_SUMMARIES[tier]
    d_total = s["total"]["direct"]["mean"]
    t_total = s["total"]["task"]["mean"]
    delta = s["total"]["delta"]
    pct = s["total"]["pct"]
    overheads.append(delta)
    print(f"  {tier:<10} {d_total:>14.0f} {t_total:>14.0f} {delta:>+14.0f} {pct:>+11.1f}%")

# Check if overhead is roughly constant or scaling
if len(overheads) >= 2 and all(o > 0 for o in overheads):
    ratio = overheads[-1] / overheads[0] if overheads[0] > 0 else float('inf')
    print(f"\n  Overhead ratio (complex/simple): {ratio:.2f}x")
    if ratio < 1.5:
        print("  → Overhead appears FIXED (roughly constant across tiers)")
    elif ratio < 3.0:
        print("  → Overhead appears MODERATELY SCALING")
    else:
        print("  → Overhead appears STRONGLY SCALING with complexity")
elif any(o <= 0 for o in overheads):
    negative = [t for t, o in zip(["simple", "medium", "complex"], overheads) if o <= 0]
    print(f"\n  WARNING: Non-positive overhead in tiers: {negative}")
    print("  → Task delegation may not add measurable overhead at these tiers")

---

## Strategy B: Separated Models (parent haiku, subagent sonnet)

To decompose parent vs subagent token consumption, we need them on different models
so `modelUsage` shows separate entries.

**Approach:** Use `--agents` CLI flag (if supported) to configure a named subagent
on sonnet, then instruct the parent to delegate to that agent.

**Fallback:** If `--agents` is not supported, instruct the parent to specify
`model: "sonnet"` in the Task tool call directly.

We test viability first before running the full experiment.

In [ ]:
# Strategy B: Viability test
# Test whether we can get separated model usage entries

def run_claude_with_agents(prompt, agents_json, n=1, label=""):
    """Run claude -p with --agents flag for model separation."""
    results = []
    for i in range(n):
        session_id = str(uuid.uuid4())
        cmd = [
            "claude", "-p", prompt,
            "--session-id", session_id,
            "--output-format", "json",
            "--model", "haiku",
            "--permission-mode", "default",
            "--agents", json.dumps(agents_json),
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0 and not result.stdout.strip():
            return {"error": result.stderr, "approach": "--agents flag"}

        output = json.loads(result.stdout) if result.stdout.strip() else {"error": result.stderr}
        usage = extract_usage(type("CR", (), {"output": output})())
        usage["run"] = i + 1
        usage["returncode"] = result.returncode
        results.append(usage)
        print(f"  [{label}] Run {i+1}/{n}: models={usage['models']} "
              f"in={usage['inputTokens']} out={usage['outputTokens']}")
    return results


def run_claude_task_with_model(prompt, model="sonnet", n=1, label=""):
    """Fallback: instruct parent to specify model in Task tool call."""
    task_prompt = (
        f"Use the Task tool to perform this task. "
        f"Set subagent_type to 'general-purpose', model to '{model}', and the prompt to: "
        f"'{prompt}' "
        f"Then return whatever the task returns."
    )
    results = []
    for i in range(n):
        cr = run_claude(task_prompt)
        usage = extract_usage(cr)
        usage["run"] = i + 1
        usage["returncode"] = cr.returncode
        results.append(usage)
        print(f"  [{label}] Run {i+1}/{n}: models={usage['models']} "
              f"in={usage['inputTokens']} out={usage['outputTokens']}")
    return results


# --- Viability test: approach 1 (--agents flag) ---
print("STRATEGY B — VIABILITY TEST")
print("=" * 60)

print("\nApproach 1: --agents flag")
agents_config = {"worker": {"model": "sonnet"}}
approach1 = run_claude_with_agents(
    "Use the worker agent to answer: What is 2+2? Reply with just the number.",
    agents_config,
    n=1,
    label="viability/agents-flag"
)

agents_flag_viable = False
if isinstance(approach1, dict) and "error" in approach1:
    print(f"  --agents flag NOT supported: {approach1['error'][:200]}")
elif isinstance(approach1, list) and len(approach1) > 0:
    if len(approach1[0].get("models", [])) > 1:
        agents_flag_viable = True
        print(f"  --agents flag VIABLE: {approach1[0]['models']}")
    else:
        print(f"  --agents flag accepted but only 1 model in usage: {approach1[0]['models']}")

# --- Viability test: approach 2 (model in Task tool call) ---
print("\nApproach 2: model='sonnet' in Task tool call")
approach2 = run_claude_task_with_model(
    "What is 2+2? Reply with just the number.",
    model="sonnet",
    n=1,
    label="viability/task-model"
)

task_model_viable = False
if isinstance(approach2, list) and len(approach2) > 0:
    if len(approach2[0].get("models", [])) > 1:
        task_model_viable = True
        print(f"  Task model approach VIABLE: {approach2[0]['models']}")
    else:
        print(f"  Only 1 model in usage: {approach2[0]['models']}")
        print(f"  Model usage detail: {approach2[0].get('model_usage', {})}")

# --- Summary ---
print(f"\n{'='*60}")
print("VIABILITY SUMMARY")
print(f"{'='*60}")
print(f"  --agents flag:       {'VIABLE' if agents_flag_viable else 'NOT VIABLE'}")
print(f"  Task model param:    {'VIABLE' if task_model_viable else 'NOT VIABLE'}")

STRATEGY_B_VIABLE = agents_flag_viable or task_model_viable
STRATEGY_B_METHOD = "agents_flag" if agents_flag_viable else "task_model" if task_model_viable else None
print(f"  Strategy B feasible: {STRATEGY_B_VIABLE} (method: {STRATEGY_B_METHOD})")

In [ ]:
# Strategy B: Full experiment (conditional on viability)

STRATEGY_B_RESULTS = {}

if not STRATEGY_B_VIABLE:
    print("Strategy B NOT VIABLE — skipping full experiment.")
    print("Cannot decompose parent vs subagent token consumption.")
else:
    print(f"STRATEGY B — FULL EXPERIMENT (method: {STRATEGY_B_METHOD})")
    print("=" * 60)

    for tier in ["simple", "medium", "complex"]:
        base_prompt = PROMPTS[tier]["direct"]
        print(f"\n--- {tier.upper()} tier ---")

        if STRATEGY_B_METHOD == "agents_flag":
            task_prompt = f"Use the worker agent to: {base_prompt}"
            results = run_claude_with_agents(
                task_prompt, agents_config, n=N_RUNS,
                label=f"stratB/{tier}"
            )
        else:
            results = run_claude_task_with_model(
                base_prompt, model="sonnet", n=N_RUNS,
                label=f"stratB/{tier}"
            )

        if isinstance(results, list):
            STRATEGY_B_RESULTS[tier] = results

            # Show per-model breakdown
            for r in results:
                for model, usage in r.get("model_usage", {}).items():
                    print(f"    Run {r['run']} {model}: "
                          f"in={usage.get('inputTokens',0)} "
                          f"out={usage.get('outputTokens',0)} "
                          f"cache={usage.get('cacheReadInputTokens',0)}")

    # Summary: parent vs subagent token split
    if STRATEGY_B_RESULTS:
        print(f"\n{'='*74}")
        print("STRATEGY B — PARENT vs SUBAGENT TOKEN DECOMPOSITION")
        print(f"{'='*74}")
        print(f"  {'Tier':<10} {'Parent Input':>14} {'Parent Out':>12} "
              f"{'Sub Input':>12} {'Sub Out':>12}")
        print(f"  {'-'*10} {'-'*14} {'-'*12} {'-'*12} {'-'*12}")

        for tier in ["simple", "medium", "complex"]:
            if tier not in STRATEGY_B_RESULTS:
                continue
            results = STRATEGY_B_RESULTS[tier]
            # Aggregate per-model across runs
            model_totals = {}
            for r in results:
                for model, usage in r.get("model_usage", {}).items():
                    if model not in model_totals:
                        model_totals[model] = {"inputTokens": [], "outputTokens": []}
                    model_totals[model]["inputTokens"].append(usage.get("inputTokens", 0))
                    model_totals[model]["outputTokens"].append(usage.get("outputTokens", 0))

            models = sorted(model_totals.keys())
            if len(models) >= 2:
                # Assume first (alphabetically) is haiku (parent), second is sonnet (subagent)
                parent_model = [m for m in models if "haiku" in m]
                sub_model = [m for m in models if "haiku" not in m]
                if parent_model and sub_model:
                    pm, sm = parent_model[0], sub_model[0]
                    pi = statistics.mean(model_totals[pm]["inputTokens"])
                    po = statistics.mean(model_totals[pm]["outputTokens"])
                    si = statistics.mean(model_totals[sm]["inputTokens"])
                    so = statistics.mean(model_totals[sm]["outputTokens"])
                    print(f"  {tier:<10} {pi:>14.0f} {po:>12.0f} {si:>12.0f} {so:>12.0f}")
            elif len(models) == 1:
                print(f"  {tier:<10}  (single model: {models[0]} — cannot decompose)")

In [ ]:
# Variance analysis
# Check stdev, CV%, and whether first run differs from subsequent (cache warming)

print("VARIANCE ANALYSIS")
print("=" * 74)

for tier in ["simple", "medium", "complex"]:
    print(f"\n--- {tier.upper()} ---")
    for pattern in ["direct", "task"]:
        results = ALL_RESULTS[tier][pattern]
        for metric in ["inputTokens", "outputTokens", "cacheReadInputTokens", "total"]:
            s = summarize(results, metric)
            cv = (s["stdev"] / s["mean"] * 100) if s["mean"] > 0 else 0
            print(f"  {pattern:>6s}/{metric:<24s} mean={s['mean']:>8.0f}  "
                  f"stdev={s['stdev']:>6.0f}  CV={cv:>5.1f}%  "
                  f"range=[{s['min']:.0f}, {s['max']:.0f}]")

# Cache warming: compare first run vs mean of subsequent runs
print(f"\n{'='*74}")
print("CACHE WARMING — First run vs subsequent")
print(f"{'='*74}")

for tier in ["simple", "medium", "complex"]:
    for pattern in ["direct", "task"]:
        results = ALL_RESULTS[tier][pattern]
        if len(results) < 2:
            continue
        first_cache = results[0]["cacheReadInputTokens"]
        rest_cache = statistics.mean([r["cacheReadInputTokens"] for r in results[1:]])
        first_input = results[0]["inputTokens"]
        rest_input = statistics.mean([r["inputTokens"] for r in results[1:]])
        print(f"  {tier}/{pattern:>6s}: "
              f"cache first={first_cache:.0f} rest_mean={rest_cache:.0f}  "
              f"input first={first_input:.0f} rest_mean={rest_input:.0f}")

In [ ]:
# Cost impact analysis
# Per-invocation cost and extrapolated to 1K invocations

print("COST IMPACT ANALYSIS (Haiku 4.5 pricing)")
print("=" * 74)
print(f"  {'Tier':<10} {'Direct $/inv':>14} {'Task $/inv':>14} "
      f"{'Overhead $/inv':>16} {'Overhead/1K':>14}")
print(f"  {'-'*10} {'-'*14} {'-'*14} {'-'*16} {'-'*14}")

total_direct_cost = 0
total_task_cost = 0

for tier in ["simple", "medium", "complex"]:
    costs = cost_estimate(ALL_SUMMARIES[tier])
    d_cost = costs["direct"]["total"]
    t_cost = costs["task"]["total"]
    overhead = costs["overhead"]
    per_1k = overhead * 1000

    total_direct_cost += d_cost
    total_task_cost += t_cost

    print(f"  {tier:<10} ${d_cost:>13.6f} ${t_cost:>13.6f} "
          f"${overhead:>+15.6f} ${per_1k:>+13.4f}")

print(f"  {'-'*10} {'-'*14} {'-'*14} {'-'*16} {'-'*14}")
total_overhead = total_task_cost - total_direct_cost
print(f"  {'TOTAL':<10} ${total_direct_cost:>13.6f} ${total_task_cost:>13.6f} "
      f"${total_overhead:>+15.6f} ${total_overhead*1000:>+13.4f}")

# Breakdown by token type for the largest tier
print(f"\nCost breakdown (complex tier):")
complex_costs = cost_estimate(ALL_SUMMARIES["complex"])
for pattern in ["direct", "task"]:
    c = complex_costs[pattern]
    print(f"  {pattern}: input=${c['input']:.6f}  output=${c['output']:.6f}  "
          f"cache=${c['cache_read']:.6f}  total=${c['total']:.6f}")

---

## Conclusions

Tested on Claude Code, 2026-02-19. Haiku 4.5 parent, 5 runs per experiment.

### Framing

Task subagents are the easy default — Claude Code uses them internally, they require no
subprocess management, and most tooling targets them. But direct `claude -p` is strictly
more powerful: you control the prompt, model, session, permission mode, and can parallelize
via `subprocess.Popen`. The question is whether that power costs more or less.

### Hypothesis verdicts

| # | Hypothesis | Verdict | Evidence |
|---|-----------|---------|----------|
| H1 | Direct `claude -p` is cheaper than Task subagents | **PROVED** | Direct uses 59% fewer tokens (simple), 43% fewer (medium), 13% fewer (complex) |
| H2 | The savings are fixed or scale with complexity | **FIXED savings** | Direct saves ~40K tokens at simple/medium tiers. Gap narrows at complex tier as both patterns converge on similar total work |
| H3 | We can decompose where Task subagent tokens go | **PROVED** | `model` param in Task tool call produces separate modelUsage entries. Parent overhead is ~56K cache + 424–828 output tokens per delegation |

### Key findings

1. **Direct `claude -p` is both more powerful and cheaper.** Task subagents pay a delegation
   tax: the parent reads its system prompt, generates Task tool call JSON, waits, then reads
   the result and generates a response. Direct invocation skips all of this.

   | Tier | Task (easy) | Direct (powerful) | Savings | % cheaper |
   |------|------------|-------------------|---------|-----------|
   | Simple | 67K tokens | 27K tokens | −40K | 59% |
   | Medium | 96K tokens | 55K tokens | −41K | 43% |
   | Complex | 97K tokens | 85K tokens | −12K | 13% |

2. **The delegation tax is structurally fixed.** The subagent must read its own system prompt
   (~14K–56K cache tokens) and the parent must generate the Task tool call + process the
   result (~424–828 output tokens). This cost is independent of task complexity. As tasks
   get more complex, both patterns converge because the actual work dominates.

3. **The tax is cheap in dollar terms.** At haiku pricing, the delegation tax is ~$0.003/invocation
   ($3/1K invocations), dominated by cache reads at $0.10/M tokens. For workflows like
   `run_critics.py` running 13 parallel critics, switching from Task subagents to direct
   `claude -p` would save ~$0.04 per batch — negligible.

4. **Where the delegation tokens go (Strategy B decomposition):**
   - **Parent overhead**: ~56K cache (system prompt re-read for result turn) + 424–828
     output tokens (Task tool call JSON + final response framing)
   - **Subagent system prompt**: 14K–60K cache tokens (scales with tool count, not task complexity)
   - **Subagent fresh input**: 3–5 tokens (negligible)

5. **`--agents` flag does not separate models.** Accepted silently but the agent answered
   directly without spawning a worker. The `model` parameter in the Task tool call is the
   viable approach for model separation.

### Implications for chaos-theory workflows

- **`run_critics.py` / `run_resolvers.py`** already use direct `claude -p` — this is the
  right choice for both power (full control over prompt, model, parallelism) and efficiency
  (no delegation tax).
- **Task subagents** are appropriate when ease of use matters more than efficiency — interactive
  skills, ad-hoc delegation from within a session, cases where subprocess management is
  impractical.
- **The power gap matters more than the token gap.** Direct `claude -p` enables prompt
  engineering, model selection, session isolation, structured JSON output, and subprocess-level
  parallelism. The 13–59% token savings are a bonus on top of these capabilities, not the
  primary reason to prefer direct invocation.
- **No urgency to migrate.** The $0.003/invocation delegation tax is negligible for current
  workflow volumes. Migrate to direct `claude -p` when you need the power, not to save tokens.

In [ ]:
# Raw data dump for reproducibility

print("RAW DATA DUMP")
print("=" * 40)

raw_data = {
    "strategy_a": {},
    "strategy_b": {},
    "summaries": {},
}

for tier in ["simple", "medium", "complex"]:
    raw_data["strategy_a"][tier] = {
        "direct": ALL_RESULTS[tier]["direct"],
        "task": ALL_RESULTS[tier]["task"],
    }
    raw_data["summaries"][tier] = {
        metric: {
            "direct": ALL_SUMMARIES[tier][metric]["direct"],
            "task": ALL_SUMMARIES[tier][metric]["task"],
            "delta": ALL_SUMMARIES[tier][metric]["delta"],
            "pct": ALL_SUMMARIES[tier][metric]["pct"],
        }
        for metric in ["inputTokens", "outputTokens", "cacheReadInputTokens", "total", "num_turns"]
    }

if STRATEGY_B_RESULTS:
    raw_data["strategy_b"] = STRATEGY_B_RESULTS

print(json.dumps(raw_data, indent=2, default=str))